In [ ]:
import os
import json
import pandas as pd

# Replace this with your actual path
BASE_DIR = "../models"

# Output lists
complete_configs = []
incomplete_configs = []

# Traverse all subdirectories
for subdir, _, files in os.walk(BASE_DIR):
    config_path = None
    has_test_results = any(f.startswith("test-results") and f.endswith(".csv") for f in files)

    for f in files:
        if f.startswith("config") and f.endswith(".json"):
            config_path = os.path.join(subdir, f)
            break

    if config_path:
        try:
            with open(config_path, 'r') as json_file:
                config_data = json.load(json_file)
        except Exception as e:
            print(f"Error reading {config_path}: {e}")
            continue

        config_data["training_status"] = "complete" if has_test_results else "incomplete"
        config_data["config_path"] = config_path

        if has_test_results:
            complete_configs.append(config_data)
        else:
            incomplete_configs.append(config_data)

# Convert to DataFrames
df_complete = pd.DataFrame(complete_configs)
df_incomplete = pd.DataFrame(incomplete_configs)

# Save results for inspection
df_complete.to_csv("complete_configs.csv", index=False)
df_incomplete.to_csv("incomplete_configs.csv", index=False)

# Display summary
print("Complete Trainings:", len(df_complete))
print("Incomplete Trainings:", len(df_incomplete))
print("All unique config keys:", set().union(*(d.keys() for d in complete_configs + incomplete_configs)))
